<!-- pick up here -->


### Step 4 — Build the X-ray scattering probability map (Cell 13)

This step converts the HI4PI N_H map into a sky map of X-ray scattering probability, using the analytical small-angle scattering model from Draine (2003).

#### Physical model

In the regime where the dimensionless size parameter `x = (4πa/λ) sin(θ/2) << 1` (valid for small grain sizes, high photon energies, or small scattering angles — the "Rayleigh–Gans" limit), the total scattering cross section simplifies to:

```
σ_scatter = (6.3×10⁻¹¹) · (2Z/M)² · (ρ_grain/3)² · (a/0.1μm)⁴ · (E/keV)⁻¹ · (F(E)/Z)² cm²
```

where Z is the nuclear charge of the dominant grain material, M is atomic mass, ρ_grain is grain material density, a is grain radius, E is photon energy, and F(E) is the atomic scattering factor (≈ Z far from resonances).

The dust grain number column density N_d relates to the hydrogen column density via:
```
N_d = 0.01 · N_H · m_p / m_grain    where m_grain = (4π/3) · a³ · ρ_grain
```

Combining these, the **scattering optical depth** at reference values (a=0.1μm, ρ=3 g/cm³) simplifies to:
```
τ_scatter = N_d · σ_scatter = 8.4×10⁻²³ · N_H · (a/0.1μm) · (ρ_grain/3) · (E_keV)⁻¹
```

The **photoelectric absorption optical depth** (cross section averaged over standard ISM composition) is:
```
τ_abs = 2.4×10⁻²² · N_H · (E_keV)⁻³
```

The ratio τ_scatter/τ_abs ≈ 0.35·E_keV², meaning:
- At **E = 1 keV**: absorption is ~3× stronger than scattering — X-rays are absorbed before forming a bright halo
- At **E = 2 keV**: scattering is ~1.4× stronger than absorption — halos are more detectable
- At **E ≥ 3 keV**: scattering strongly dominates, but halos become geometrically smaller and harder to resolve

The **scattering probability** for a photon traversing the full column is:
```
P_scatter = 1 − e^(−τ_scatter)
```

#### Implementation

The function `get_tau_scattering_and_absorption(a, rho_grain, E_keV)` loads `NHI_HPX.fits`, maps N_H values to their HEALPix pixel indices, clips negative values, and returns `tau_scattering` and `tau_absorption` as NumPy arrays of shape (12,582,912,) indexed in galactic HEALPix RING pixels at NSIDE=1024.

To visualize these as equirectangular images, a 1400×700 pixel grid in (RA, Dec) is constructed, converted to galactic (l, b) via astropy SkyCoord, then to HEALPix pixel indices via `hp.ang2pix`, and finally used to look up τ values. The grayscale image assigns darker tones to higher τ (higher scattering probability) using thresholds that correspond to round scattering probability values (10%, 20%, ..., 85%).

**Measured sky statistics at E=1 keV, NSIDE=1024:**
- τ_max = 2.015 (P_scatter = 86.7%), τ_mean = 0.103
- Sky fraction with τ > 1 (P_scatter > 63%): 0.6% — this is essentially the Galactic plane
- Sky fraction with τ > 0.10536 (P_scatter > 10%): 23.6%

### Step 5 — Calculate overlap percentages (Cell 15)

For each event, the fraction of the 90% credible region that falls within the high-scattering zone is computed:

```python
scattering_mask = tau_scattering > tau_threshold    # boolean, galactic HEALPix pixels
overlap_pixels  = np.sum(confidence_map & scattering_mask)
overlap_pct     = 100 * overlap_pixels / np.sum(confidence_map)
```

This pixel-wise AND operation works correctly because:
- Both arrays are at the same NSIDE (1024)
- The coordinate conversion pipeline in Step 4 ensures galactic HEALPix pixel indices in `scattering_mask` correspond to the same sky positions as equatorial HEALPix pixel indices in `confidence_map` (after conversion)

Results are sorted descending by overlap percentage and saved to:
- `bbh_overlap_percentages_>=0.5_scattering_probability.csv` (τ threshold = 0.69315, P_scatter ≥ 50%)
- `bbh_overlap_percentages_>=0.1_scattering_probability.csv` (τ threshold = 0.10536, P_scatter ≥ 10%)

**Notable result**: GW200224_222234 shows **100% overlap** at the ≥10% threshold. Its 90% credible region covers only ~50 deg² — exceptionally well-localized for a GW event — and lies entirely within the high-scattering part of the sky (near the Galactic plane).

### Step 6 — Probability of detection across all events (Cells 15.5.1–15.5.8)

Treating each event independently, the probability that **at least one** event has its 90% credible region fully (or substantially) overlapping a high-scattering zone is:

```
P(at least one success) = 1 − ∏ᵢ (1 − overlap_i / 100)
```

**Numerical issue**: direct floating-point multiplication of 167 small factors (each ≤1) underflows to exactly 0.0 in float64, making the complement 1 − 0 = 1 uninformative.

**Solution**: compute in log space using the identity log(∏ xᵢ) = Σ log(xᵢ):

```python
log_prob_total_failure = Σᵢ log(max(1 − overlap_i/100, 1e-10))
# clamp to 1e-10 to handle the 100% overlap event (log(0) = −∞)
prob_total_failure = exp(log_prob_total_failure)
prob_success = −np.expm1(log_prob_total_failure)
# np.expm1(x) = e^x − 1 with full precision near x=0,
# avoiding catastrophic cancellation when subtracting two nearly-equal floats
```

Sensitivity analysis is performed by successively halving the event list (all 167 → top 84 → top 42 → top 21) and by removing the dominant event (GW200224_222234) to assess how much one outlier drives the result. The full results table is documented in the Methodology markdown cell of the notebook.

---

## Outputs

| File | Description |
|---|---|
| `skymaps/*.fits(.gz)` | 167 individual HEALPix skymap files, one per BBH event |
| `{year}_bbh_events_90%_regions.png` | Per-year equirectangular plots of all 90% credible regions (years: 2015, 2017, 2019, 2020, 2023, 2024) |
| `scattering_probability_of_x-rays_>=_0.1.png` | Full-sky grayscale map of X-ray scattering probability at E=1 keV |
| `bbh_chunk_N_with_overlay.png` | 17 plots of 10 events each, overlaid on the dust/scattering background |
| `bbh_overlap_percentages_>=0.5_scattering_probability.csv` | Overlap percentages at ≥50% scattering threshold, sorted descending |
| `bbh_overlap_percentages_>=0.1_scattering_probability.csv` | Overlap percentages at ≥10% scattering threshold, sorted descending |
